In [1]:
import os
import random
import numpy as np

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## input onehot
IN_CH = 4

## encoder stats
DNA_CHANNELS = 112
DOWNRES_CHANNELS = 32

## geometry
L_bp = 2**13
STEP = 4
RES = 2**(STEP+1)
T = L_bp // RES
CRES = RES*2**4
T_dim = DNA_CHANNELS + DOWNRES_CHANNELS * STEP # 240
CT = L_bp // CRES
head_count = 4



## attention setup
q_count = 4
kv_count = 1

assert T == 2**8

print(RES)
print(CRES)
print("D: ", L_bp)
print("embedding resolution: ", RES)
print("token number: ", T)
print("Ctoken number: ", CT)
print("token dim: ", T_dim)
print("numer of heads: ", head_count)

device: cpu
32
512
D:  8192
embedding resolution:  32
token number:  256
Ctoken number:  16
token dim:  240
numer of heads:  4


In [9]:
class StandarizedConv1D(nn.Module):
    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size,
                 stride = 1,
                 padding = None) -> None:

      super().__init__()
      self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size))
      self.bias = nn.Parameter(torch.zeros(out_channels))
      self.stride = stride
      if padding == None:
        self.padding = (kernel_size-1) // 2
      else:
        self.padding = padding

    def forward(self, x):
      """
      x: [B, C, S]
      returns: [B, C_new, S]
      """

      w = self.weight
      w = w - w.mean(dim=(1,2), keepdim=True)
      w = w / torch.sqrt(w.var(dim=(1,2), keepdim=True, unbiased=False) + 1e-5)
      return F.conv1d(x, w, self.bias, self.stride, self.padding)



class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, width=5):
        super().__init__()
        self.norm = nn.RMSNorm(in_ch)

        if width == 1:
            self.op = nn.Linear(in_ch, out_ch)
        else:
            self.op = StandarizedConv1D(in_ch, out_ch, kernel_size=width)

    def forward(self, x):
      """
      x: [B, C, S]
      return: [B, C_new, S]
      """
      x = x.transpose(1, 2)
      x = self.norm(x)
      x = F.gelu(x)

      ## x: [B, S, C]
      if isinstance(self.op, nn.Linear):
        x = self.op(x)
        x = x.transpose(1, 2)
      else:
        x = x.transpose(1, 2)
        x = self.op(x)

      return x


class Encoder(nn.Module):
    def __init__(
        self,
        in_ch=4,
        dna_channels=112,
        DNA_width = 15,
        downres_channels=32,
        num_stages=5,   # for [1,2,4,8,16]
    ):
        super().__init__()

        blocks = []

        for i in range(num_stages):
            block = []
            if i == 0:
              ## DNA Block
              dna1 = nn.Conv1d(in_ch,
                                    dna_channels,
                                    DNA_width,
                                    1, 7)
              dna2 = ConvBlock(dna_channels, dna_channels)
              block.append(dna1)
              block.append(dna2)
            else:
              ## Downres_block
              downres_in  = dna_channels + (i - 1) * downres_channels
              downres_out = dna_channels + (i) * downres_channels
              downres1 = ConvBlock(downres_in, downres_out)
              downres2 = ConvBlock(downres_out, downres_out)
              block.append(downres1)
              block.append(downres2)

            block = nn.ModuleList(block)
            blocks.append(block)

        self.blocks = nn.ModuleList(blocks)

    def dna_embedder(self, x, block):
        out = block[0](x)
        return out + block[1](out)

    def downres_block(self, x, block):
        out = block[0](x)
        # channel padding skip connection
        pad_ch = out.shape[1] - x.shape[1]
        out = out + F.pad(x, (0, 0, 0, pad_ch))
        return out + block[1](out)

    def forward(self, x):
        """
        x: [B, C, S]
        returns: [B, S, C]
        """

        intermediates = {}
        for i, bin_size in enumerate([1, 2, 4, 8, 16]):
          if i == 0:
            x = self.dna_embedder(x, self.blocks[i])
          else:
            x = self.downres_block(x, self.blocks[i])

          intermediates[f"bin_size_{bin_size}"] = x

          x = F.max_pool1d(x, 2, 2)

        return x.transpose(1,2), intermediates

In [3]:
from math import tanh


class TransformerBlock(nn.Module):
  def __init__(self,
               input_channels=240,
               head_count=4,
               q_dim=32,
               v_dim=40,
               pairwise_transformer_channels=32,
               downsample_ratio=4) -> None:
      super().__init__()
      self.C = input_channels
      self.head_count = head_count
      self.q_dim = self.k_dim = q_dim
      self.v_dim = v_dim
      self.R = downsample_ratio

      ## attention
      self.attnnorm1 = nn.RMSNorm(input_channels)
      self.attnnorm2 = nn.RMSNorm(input_channels)
      self.qnorm = nn.LayerNorm(self.q_dim)
      self.knorm = nn.LayerNorm(self.q_dim)
      self.vnorm = nn.LayerNorm(self.v_dim)
      self.q_proj = nn.Linear(self.C, self.head_count * self.q_dim)
      self.k_proj = nn.Linear(self.C, self.k_dim)
      self.v_proj = nn.Linear(self.C, self.v_dim)
      self.out_proj = nn.Linear(self.head_count * self.v_dim, self.C)
      self.fc1 = nn.Linear(input_channels, self.C*2)
      self.fc2 = nn.Linear(self.C*2, self.C)

      ## linear
      self.mlpnorm1 = nn.RMSNorm(self.C)
      self.mlpnorm2 = nn.RMSNorm(self.C)

      ## attention bias
      self.attnbiasnorm = nn.RMSNorm(pairwise_transformer_channels)
      self.attnbiaslinear_proj = nn.Linear(pairwise_transformer_channels,
                                           head_count, bias=False)





  def apply_rope(self, x):
    # x: [B, H, T, d]
    B, H, T, d = x.shape
    d2 = d // 2

    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    positions = torch.arange(T, device = x.device)
    inv_freq = 1.0 / (1e4) ** (torch.arange(d2, device = x.device) / (d2))

    theta = positions[:, None] * inv_freq[None, :] # [T, d2]
    cos = torch.cos(theta)[None, None, :, :]
    sin = torch.sin(theta)[None, None, :, :]


    out_even = x_even * cos - x_odd * sin
    out_odd = x_even * sin + x_odd * cos

    out = torch.empty_like(x)

    out[..., 0::2] = out_even
    out[..., 1::2] = out_odd

    return out


  def forward(self, x_tok:torch.Tensor, pair_x:torch.Tensor = None) -> torch.Tensor:
    """
    x: [B, S, C], pair_x: [B, P, P, F]

    returns: [B, S, C]
    """
    attention_bias = None

    if pair_x is not None:
      b = pair_x
      b = F.gelu(self.attnbiasnorm(b))
      b = self.attnbiaslinear_proj(b)
      b = b.repeat_interleave(self.R, dim=1)
      b = b.repeat_interleave(self.R, dim=2)
      attention_bias = b.permute(0, 3, 1, 2)

    ## Attention_bias: [B, H, S, S,]

    ## B, S, C
    x = x_tok

    x_att_in = self.attnnorm1(x)
    B, T, _ = x_tok.shape
    assert x_tok.shape[-1] == self.C

    q_raw = self.q_proj(x_att_in) ## B, T, h* d_q
    k_raw = self.k_proj(x_att_in) ## B, T, 1* d_q
    v_raw = self.v_proj(x_att_in) ## B, T, 1* d_v

    q = q_raw.view(B, T, self.head_count, self.q_dim) ## B, T, 4, d_q
    k = k_raw.view(B, T, 1, self.k_dim) ## B, T, 1, d_q
    v = v_raw.view(B, T, 1, self.v_dim) ## B, T, 1, d_v

    q = self.qnorm(q)
    k = self.knorm(k)
    v = self.vnorm(v)


    q = q.permute(0, 2, 1, 3) ## B, H, T, d_q
    k = k.permute(0, 2, 1, 3) ## B, H, T, d_k
    v = v.permute(0, 2, 1, 3)  ## B, H, T, d_v

    q = self.apply_rope(q)
    k = self.apply_rope(k)
    kt = k.transpose(-2, -1)
    logits = q @ kt / (self.q_dim ** .5)

    if attention_bias is not None:
      logits += attention_bias

    logits = torch.tanh(logits / 5.0) * 5.0
    weights = torch.softmax(logits, dim = -1)

    y = (weights @ v)
    y_flat = y.permute(0, 2, 1, 3).reshape(B, T, self.head_count * self.v_dim)
    y = self.out_proj(y_flat)
    x_att_out = self.attnnorm2(y)
    x = x + x_att_out

    mlp_in = self.mlpnorm1(x)
    mlp_out = self.mlpnorm2(self.fc2(F.relu(self.fc1(mlp_in))))

    final_out = mlp_out + x
    assert final_out.shape == x_tok.shape
    assert x.shape[-1] == self.C

    return final_out

In [4]:
from itertools import count
## Pairwise block

class PairwiseBlock(nn.Module):
  def __init__(self,
               head_count = 16,
               transformer_channels = 32,
               downsample_ratio = 4,
               sequence_length = 256,
               input_channels = 240,
               feature_size = 16) -> None:
     super().__init__()

     self.C = input_channels
     self.K = transformer_channels
     self.H = head_count
     self.R = downsample_ratio
     self.P = sequence_length // self.R
     self.register_buffer("pos_features",
                          self.central_mask_features(self.P, feature_size),
                          persistent=False)


     self.q_r_bias = nn.Parameter(torch.zeros(1, 1, head_count, transformer_channels))
     self.k_r_bias = nn.Parameter(torch.zeros(1, 1, head_count, transformer_channels))


     self.avgPool = nn.AvgPool1d(kernel_size=self.R, stride=self.R)
     self.norm = nn.RMSNorm(input_channels)
     self.q_proj = nn.Linear(input_channels, head_count * transformer_channels, bias=False)
     self.k_proj = nn.Linear(input_channels, head_count * transformer_channels, bias=False)
     self.linear_pos = nn.Linear(feature_size, head_count * transformer_channels)
     self.linear_proj_q = nn.Linear(input_channels, transformer_channels, bias = False)
     self.linear_proj_k = nn.Linear(input_channels, transformer_channels, bias = False)
     self.linear_proj = nn.Linear(head_count, transformer_channels)



     self.q_proj_rowattention = nn.Linear(transformer_channels, 1*transformer_channels, bias=False)
     self.k_proj_rowattention = nn.Linear(transformer_channels, 1*transformer_channels, bias=False)
     self.v_proj_rowattention = nn.Linear(transformer_channels, 1*transformer_channels)
     self.norm_rowattention = nn.RMSNorm(transformer_channels)

     self.mlp_norm = nn.RMSNorm(transformer_channels)
     self.mlp_linear1 = nn.Linear(transformer_channels, 2*transformer_channels)
     self.mlp_linear2 = nn.Linear(2*transformer_channels, transformer_channels)

  def central_mask_features(self, P, feature_size):

      assert feature_size % 2 == 0

      relative_positions = torch.arange(2 * P - 1) - (P - 1)

      half = feature_size // 2

      geom = torch.exp(
          torch.linspace(
              math.log(1.0),
              math.log(float(P - half + 1)),
              steps=half
          )
      )

      center_widths = torch.arange(half, dtype=torch.float32) + geom

      embeddings = (center_widths[None, :] >
                    relative_positions.abs().float()[:, None]).float()

      signed_embeddings = torch.sign(relative_positions.float())[:, None] * embeddings

      pos_features = torch.cat([embeddings, signed_embeddings], dim=-1)

      return pos_features

  def pair_mlp_block(self, x: torch.Tensor) -> torch.Tensor:
    """
    x: [B, P, P, F]
    ret: [B, P, P, F]
    """
    x = self.mlp_norm(x)
    x = F.relu(self.mlp_linear1(x))
    x = self.mlp_linear2(x)

    return x

  def row_attention_block(self, x: torch.Tensor) -> torch.Tensor:
    """
    x: [B, P, P, F]
    returns: [B, P, P, F]
    """
    B, P, P, fin = x.shape

    x = self.norm_rowattention(x)

    q_raw = self.q_proj_rowattention(x) # [B, P, P, F]
    k_raw = self.k_proj_rowattention(x) # [B, P, P, F]
    v_raw = self.v_proj_rowattention(x)

    q = q_raw
    k = k_raw
    v = v_raw

    logits = torch.einsum('bpPf,bpkf->bpPk', q, k) / math.sqrt(fin)
    logits = F.softmax(logits, dim = -1)
    x = torch.einsum('bpPk,bpkf->bpPf', logits, v)

    return x

  def relative_shift(self, x: torch.Tensor) -> torch.Tensor:
      """
      x: [B, Q, 2Q-1, H]
      returns: [B, Q, Q, H]
      """
      B, Q, R, H = x.shape
      assert R == 2 * Q - 1

      # pad a dummy column so we can do the shift via reshape
      x = F.pad(x, (0, 0, 1, 0))          # [B, Q, 2Q, H]
      x = x.view(B, R + 1, Q, H)          # [B, 2Q, Q, H]
      x = x[:, 1:, :, :]                  # [B, 2Q-1, Q, H]
      x = x.view(B, Q, R, H)              # [B, Q, 2Q-1, H]

      # keep only the first Q relative positions aligned to keys
      return x[:, :, :Q, :]               # [B, Q, Q, H]


  def seq_to_pair_block(self,
                        x : torch.Tensor
                        ) -> torch.Tensor:
     """
     x : [B, S, C]
     rets: [B, P, P, F]
     """

     x = self.avgPool(x.transpose(-1, -2)) # [B, S, C] -> [B, C, S]
     x = x.transpose(-1, -2) # [B, P, C]
     x = self.norm(x)
     B, P, _ = x.shape
     q = self.q_proj(x).view(B, P, self.H, self.K) # [B, P, H, F]
     k = self.k_proj(x).view(B, P, self.H, self.K) # [B, P, H, F]
     pos_encodings = self.linear_pos(
         self.pos_features).view(
             2 * P - 1, self.H, self.K) ## [2P-1, H, F]
     rel_q = self.relative_shift(torch.einsum('bqhc,rhc->bqrh', q + self.q_r_bias, pos_encodings))
     rel_k = self.relative_shift(torch.einsum('bkhc,rhc->bkrh', k + self.k_r_bias, pos_encodings))
     logits = torch.einsum('bqhc,bkhc->bqkh', q, k) + (rel_q + rel_k.transpose(1,2)) / 2 # [B, P, P, H]

     y_q = self.linear_proj_q(F.gelu(x)) # [B, P, F]
     y_k = self.linear_proj_k(F.gelu(x)) # [B, P, F]
     out = self.linear_proj(logits) + y_q[:, :, None, :] + y_k[:, None, :, :]  # [B, P, P, F]

     return out


  def forward(self,
              sequence_input : torch.Tensor,
              pair_input : torch.Tensor | None = None
              ) -> torch.Tensor:
    """
    sequence_input : [B, S, C]
    pair_input: [B, P, P, F]
    """
    y = self.seq_to_pair_block(sequence_input)
    x = y if pair_input is None else pair_input + y

    x += self.row_attention_block(x)
    x += self.pair_mlp_block(x)

    return x

In [5]:
class TransformerTower(nn.Module):
  def __init__(self) -> None:
     super().__init__()
     self.blocks = nn.ModuleList([TransformerBlock() for _ in range(9)])

     self.pair_update_blocks = nn.ModuleList([PairwiseBlock() for _ in range(5)])

  def forward(self, x, pair_x = None):

    for i, block in enumerate(self.blocks):
      if i % 2 == 0:
        pair_x = self.pair_update_blocks[i//2](x, pair_x)

      x = block(x, pair_x = pair_x)

    return x, pair_x



In [19]:
B = 1
L = 2**13   # 8192
IN_CH = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

encoder = Encoder().to(device)

bases = torch.randint(0, 4, (B, L), device=device)
x = F.one_hot(bases, num_classes=4)
x = x.permute(0, 2, 1).float()

x, inter = encoder(x)
print("bin_size_1:", inter["bin_size_1"].shape)  # expect (B, 112, 8192)
print("bin_size_2:", inter["bin_size_2"].shape)  # expect (B, 144, 4096)

print("x:", x.shape)  # expect (B, 240, 256)
assert x.shape == (B, 256, 112 + 32 * 4)

device: cpu
bin_size_1: torch.Size([1, 112, 8192])
bin_size_2: torch.Size([1, 144, 4096])
x: torch.Size([1, 256, 240])


In [12]:
from pprint import pprint
for k in inter.values():
  pprint(k.shape)


torch.Size([1, 112, 8192])
torch.Size([1, 144, 4096])
torch.Size([1, 176, 2048])
torch.Size([1, 208, 1024])
torch.Size([1, 240, 512])


In [20]:
tt = TransformerTower()
x, x_pair = tt(x, None)

In [41]:
## decoder block
class Decoder(nn.Module):
  def __init__(self,
               intermediates,
               downres_channel = 32) -> None:
    super().__init__()
    num_stages = len(intermediates)

    self.residual_scales = nn.ParameterList(
        [nn.Parameter(torch.tensor(0.9)) for _ in range(num_stages)]
    )
    self.downres_channel = downres_channel
    self.blocks = []



    for i, bin_size in enumerate([16, 8, 4, 2, 1]):
      if bin_size == 1:
        downres_channel = 0
      block = nn.ModuleList()
      inter = intermediates[f'bin_size_{bin_size}']
      B, C, _ = inter.shape
      U = C - downres_channel
      block.append(ConvBlock(C, U))
      block.append(ConvBlock(C, U, width=1))
      block.append(ConvBlock(U, U))

      self.blocks.append(nn.ModuleList(block))

    self.blocks = nn.ModuleList(self.blocks)


  def upres(self, x, unet_skip, block, downres_channels, i):
    """
    x: [B, C, S]
    unet_skip: [B, C_new, S*2]
    """

    # U = unet_skip.shape[1]
    # tmp = block[0](x)
    # assert tmp.shape[1] + downres_channels == U

    _, C, _ = x.shape

    out = block[0](x) + x[:, :(C-downres_channels), :]


    out = out.repeat_interleave(2, dim = 2) * self.residual_scales[i]
    out += block[1](unet_skip)
    return out + block[2](out)


  def forward(self, x, intermediates):

    for i, bin_size in enumerate([16, 8, 4, 2, 1]):
      downres_channel = self.downres_channel if bin_size != 1 else 0

      x = self.upres(x,
                     intermediates[f'bin_size_{bin_size}'],
                     self.blocks[i],
                     downres_channel,
                     i)

    return x

In [42]:
m = Decoder(inter, downres_channel=32)

out = m(x.transpose(1, 2), inter)

out.shape

torch.Size([1, 112, 8192])